[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/24.%20The%20Static%20Truck%20Appointment%20System%20Problem/P24-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 24. The Static Truck Appointment System Problem

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Goal
Formulate the Static Truck Appointment System as a Mixed-Integer Programming (MIP) problem to find optimal time slot assignments that minimize total deviation penalties while respecting gate capacity constraints.

### Key assumptions
- Each truck request has a preferred time slot and priority weight
- Each time slot has a fixed processing capacity
- Deviation from preferred time incurs a penalty proportional to priority weight
- Gate imbalance (uneven load distribution) should be minimized
- All requests must be assigned to exactly one time slot

### Approach (step-by-step)
1. **Define decision variables** for request-to-slot assignments
2. **Formulate objective function** minimizing weighted deviation and gate imbalance
3. **Add assignment constraints** ensuring each request gets exactly one slot
4. **Add capacity constraints** respecting time slot processing limits
5. **Implement MIP solver** using pulp library
6. **Extract and display solution** with detailed assignment matrix
7. **Perform sensitivity analysis** on key parameters

### What to look for in the results
- Optimal assignment matrix showing which request goes to which time slot
- Total penalty value with deviation and imbalance components
- Individual request assignments with deviation calculations
- Gate utilization distribution across time slots
- Sensitivity analysis showing impact of parameter changes

### Concrete example (from the source)
The system processes 4 truck requests across 3 time slots with 2 gates. Each request has a preferred time slot and priority weight, and the goal is to minimize total weighted deviation while balancing gate utilization.

**Expected Output:**
```
Assignment Matrix:
Request → Time Slot:
   Slot1 Slot2 Slot3
Req1:    X          (pref: 0)
Req2:        X      (pref: 1)
Req3:            X  (pref: 2)
Req4:        X      (pref: 1, deviation: +1)

Total Penalty: 2.50
Deviation Penalty: 1.50
Gate Imbalance Penalty: 1.00
```

In [1]:
# Import required libraries for mathematical optimization
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully for Static Truck Appointment System optimization")

Libraries imported successfully for Static Truck Appointment System optimization


In [2]:
# Define data structures for the problem
class TruckRequest:
    """Represents a truck appointment request"""
    def __init__(self, id: int, preferred_time: int, weight: float = 1.0):
        self.id = id
        self.preferred_time = preferred_time
        self.weight = weight  # Priority weight for deviation penalty
    
    def __repr__(self):
        return f"Request({self.id}, pref_time={self.preferred_time}, weight={self.weight})"

class TimeSlot:
    """Represents a time slot for truck processing"""
    def __init__(self, id: int, capacity: int):
        self.id = id
        self.capacity = capacity  # Maximum trucks that can be processed
    
    def __repr__(self):
        return f"TimeSlot({self.id}, capacity={self.capacity})"

class Gate:
    """Represents a processing gate"""
    def __init__(self, id: int, capacity: int = 2):
        self.id = id
        self.capacity = capacity
    
    def __repr__(self):
        return f"Gate({self.id}, capacity={self.capacity})"

print("Data structures defined for truck appointment system")

Data structures defined for truck appointment system


In [ ]:
# Create the concrete example problem instance
def create_concrete_example():
    """Create the concrete example with 4 requests, 3 slots, 2 gates"""
    
    # Define truck requests with preferred times and weights
    requests = [
        TruckRequest(1, 0, 1.0),  # Request 1 prefers slot 0, weight 1.0
        TruckRequest(2, 1, 1.5),  # Request 2 prefers slot 1, weight 1.5 (higher priority)
        TruckRequest(3, 2, 1.0),  # Request 3 prefers slot 2, weight 1.0
        TruckRequest(4, 1, 0.8)   # Request 4 prefers slot 1, weight 0.8 (lower priority)
    ]
    
    # Define time slots with capacities
    time_slots = [
        TimeSlot(0, 2),  # Slot 0 can process 2 trucks
        TimeSlot(1, 3),  # Slot 1 can process 3 trucks  
        TimeSlot(2, 2)   # Slot 2 can process 2 trucks
    ]
    
    # Define processing gates
    gates = [
        Gate(0, 2),  # Gate 0 can process 2 trucks simultaneously
        Gate(1, 2)   # Gate 1 can process 2 trucks simultaneously
    ]
    
    print("Concrete Example Created:")
    print(f"  Truck Requests: {len(requests)}")
    print(f"  Time Slots: {len(time_slots)}")
    print(f"  Gates: {len(gates)}")
    
    print("\nRequests:")
    for req in requests:
        print(f"  {req}")
    
    print("\nTime Slots:")
    for slot in time_slots:
        print(f"  {slot}")
    
    print("\nGates:")
    for gate in gates:
        print(f"  {gate}")
    
    return requests, time_slots, gates

# Create the problem instance
requests, time_slots, gates = create_concrete_example()

Concrete Example Created:
  Truck Requests: 4
  Time Slots: 3
  Gates: 2

Requests:
  Request(1, pref_time=0, weight=1.0)
  Request(2, pref_time=1, weight=1.5)
  Request(3, pref_time=2, weight=1.0)
  Request(4, pref_time=1, weight=0.8)

Time Slots:
  TimeSlot(0, capacity=2)
  TimeSlot(1, capacity=3)
  TimeSlot(2, capacity=2)

Gates:
  Gate(0, capacity=2)
  Gate(1, capacity=2)


In [3]:
# MIP Model Implementation
class StaticTruckAppointmentMIP:
    """Mixed-Integer Programming model for Static Truck Appointment System"""
    
    def __init__(self, requests, time_slots, gates, alpha=1.0, beta=0.5):
        """Initialize MIP model
        
        Args:
            requests: List of truck requests
            time_slots: List of time slots
            gates: List of processing gates
            alpha: Weight for deviation penalty
            beta: Weight for gate imbalance penalty
        """
        self.requests = requests
        self.time_slots = time_slots
        self.gates = gates
        self.alpha = alpha  # Deviation penalty weight
        self.beta = beta    # Gate imbalance penalty weight
        
        self.model = None
        self.x_vars = {}    # Assignment variables: x[r,t] = 1 if request r assigned to slot t
        self.y_vars = {}    # Gate assignment variables: y[g,t] = trucks assigned to gate g at slot t
        self.delta_plus_vars = {}  # Positive deviation variables
        self.delta_minus_vars = {} # Negative deviation variables
        
        self.solution = None
    
    def build_model(self):
        """Build the MIP model"""
        # Create model
        self.model = pulp.LpProblem("Static_Truck_Appointment_System", pulp.LpMinimize)
        
        # Decision variables
        # x[r,t] = 1 if request r is assigned to time slot t
        for r, t in product(self.requests, self.time_slots):
            var_name = f"x_{r.id}_{t.id}"
            self.x_vars[(r.id, t.id)] = pulp.LpVariable(var_name, cat='Binary')
        
        # y[g,t] = number of trucks assigned to gate g at time slot t
        for g, t in product(self.gates, self.time_slots):
            var_name = f"y_{g.id}_{t.id}"
            self.y_vars[(g.id, t.id)] = pulp.LpVariable(var_name, lowBound=0, cat='Integer')
        
        # Deviation variables
        for r in self.requests:
            self.delta_plus_vars[r.id] = pulp.LpVariable(f"delta_plus_{r.id}", lowBound=0)
            self.delta_minus_vars[r.id] = pulp.LpVariable(f"delta_minus_{r.id}", lowBound=0)
        
        # Objective function
        # Minimize: alpha * weighted_deviation + beta * gate_imbalance
        
        # Weighted deviation component
        deviation_penalty = pulp.lpSum([
            r.weight * (self.delta_plus_vars[r.id] + self.delta_minus_vars[r.id])
            for r in self.requests
        ])
        
        # Gate imbalance component
        gate_imbalance = 0
        for t in self.time_slots:
            gate_loads = [self.y_vars[(g.id, t.id)] for g in self.gates]
            # Use auxiliary variables for max and min
            max_load = pulp.LpVariable(f"max_load_{t.id}", lowBound=0)
            min_load = pulp.LpVariable(f"min_load_{t.id}", lowBound=0)
            
            # Constraints for max and min
            for load in gate_loads:
                self.model += max_load >= load
                self.model += min_load <= load
            
            gate_imbalance += self.beta * (max_load - min_load)
        
        self.model += deviation_penalty + gate_imbalance, "Total_Penalty"
        
        # Constraints
        
        # 1. Each request must be assigned to exactly one time slot
        for r in self.requests:
            self.model += pulp.lpSum([self.x_vars[(r.id, t.id)] for t in self.time_slots]) == 1, f"assign_once_{r.id}"
        
        # 2. Time slot capacity constraints
        for t in self.time_slots:
            assigned_trucks = pulp.lpSum([self.x_vars[(r.id, t.id)] for r in self.requests])
            self.model += assigned_trucks <= t.capacity, f"capacity_{t.id}"
        
        # 3. Gate assignment consistency
        for t in self.time_slots:
            assigned_trucks = pulp.lpSum([self.x_vars[(r.id, t.id)] for r in self.requests])
            total_gate_capacity = pulp.lpSum([self.y_vars[(g.id, t.id)] for g in self.gates])
            self.model += total_gate_capacity == assigned_trucks, f"gate_consistency_{t.id}"
        
        # 4. Gate capacity constraints
        for g, t in product(self.gates, self.time_slots):
            self.model += self.y_vars[(g.id, t.id)] <= g.capacity, f"gate_capacity_{g.id}_{t.id}"
        
        # 5. Deviation definition constraints
        for r in self.requests:
            for t in self.time_slots:
                # t.id - r.preferred_time = delta_plus - delta_minus
                deviation = t.id - r.preferred_time
                self.model += deviation == self.delta_plus_vars[r.id] - self.delta_minus_vars[r.id], f"deviation_def_{r.id}_{t.id}"
        
        # Link deviation variables to assignment variables
        for r in self.requests:
            for t in self.time_slots:
                # If assigned to slot t, then deviation applies
                self.model += self.delta_plus_vars[r.id] <= 100 * self.x_vars[(r.id, t.id)] + 100, f"link_plus_{r.id}_{t.id}"
                self.model += self.delta_minus_vars[r.id] <= 100 * self.x_vars[(r.id, t.id)] + 100, f"link_minus_{r.id}_{t.id}"
    
    def solve(self, time_limit=60):
        """Solve the MIP model
        
        Args:
            time_limit: Time limit for solver in seconds
            
        Returns:
            bool: True if optimal solution found, False otherwise
        """
        if self.model is None:
            raise ValueError("Model not built. Call build_model() first.")
        
        # Set solver parameters
        solver = pulp.PULP_CBC_CMD(timeLimit=time_limit, msg=False)
        
        # Solve the model
        result = self.model.solve(solver)
        
        # Check solution status
        if pulp.LpStatus[self.model.status] == 'Optimal':
            self._extract_solution()
            return True
        else:
            return False
    
    def _extract_solution(self):
        """Extract solution from the solved model"""
        self.solution = {
            'assignments': {},
            'gate_assignments': {},
            'total_penalty': pulp.value(self.model.objective),
            'deviation_penalty': 0,
            'gate_imbalance_penalty': 0
        }
        
        # Extract request assignments
        for r, t in product(self.requests, self.time_slots):
            if self.x_vars[(r.id, t.id)].value() > 0.5:  # Binary variable
                self.solution['assignments'][r.id] = t.id
        
        # Extract gate assignments
        for g, t in product(self.gates, self.time_slots):
            self.solution['gate_assignments'][(g.id, t.id)] = int(self.y_vars[(g.id, t.id)].value())
        
        # Calculate penalty components
        for r in self.requests:
            assigned_slot = self.solution['assignments'][r.id]
            deviation = abs(assigned_slot - r.preferred_time)
            self.solution['deviation_penalty'] += r.weight * deviation

print("MIP model class defined successfully")

MIP model class defined successfully


In [ ]:
# Solve the concrete example using MIP
def solve_concrete_example():
    """Solve the concrete example problem"""

    print("=" * 60)
    print("STATIC TRUCK APPOINTMENT SYSTEM - MATHEMATICAL OPTIMIZATION")
    print("=" * 60)

    # Create MIP model
    mip_model = StaticTruckAppointmentMIP(requests, time_slots, gates)

    print("\n1. Building MIP model...")
    mip_model.build_model()
    print(f"   - Decision variables: {len(mip_model.x_vars)} assignment variables")
    print(f"   - Gate variables: {len(mip_model.y_vars)} gate assignment variables")
    print(f"   - Deviation variables: {len(mip_model.delta_plus_vars) + len(mip_model.delta_minus_vars)} deviation variables")

    print("\n2. Solving optimization problem...")
    if mip_model.solve():
        solution = mip_model.solution
        print("   ✓ Optimal solution found!")

        print("\n3. Solution Results:")
        print(f"   Total Penalty: {solution['total_penalty']:.2f}")
        print(f"   Deviation Penalty: {solution['deviation_penalty']:.2f}")

        print("\n4. Assignment Matrix:")
        assignment_matrix = [[0 for _ in time_slots] for _ in requests]
        for r, t in solution['assignments'].items():
            assignment_matrix[r-1][t-1] = 1

        # Display assignment matrix
        print("   Request → Time Slot:")
        print("   " + "".join(f"Slot{t.id:>4}" for t in time_slots))
        for i, req in enumerate(requests):
            row = "".join(f"{'X' if assignment_matrix[i][j] else ' ':>5}" for j in range(len(time_slots)))
            print(f"   Req{req.id}:{row} (pref: {req.preferred_time})")

        print("\n5. Detailed Assignments:")
        for req in requests:
            assigned_slot = solution['assignments'][req.id]
            deviation = abs(assigned_slot - req.preferred_time)
            print(f"   Request {req.id}: Slot {assigned_slot} (preferred: {req.preferred_time}, deviation: {deviation})")

        print("\n6. Gate Utilization:")
        for t in time_slots:
            gate_loads = [solution['gate_assignments'][(g.id, t.id)] for g in gates]
            total_load = sum(gate_loads)
            print(f"   Time Slot {t.id}: Total {total_load}/{t.capacity} trucks")
            for i, load in enumerate(gate_loads):
                if load > 0:
                    print(f"     Gate {i}: {load} trucks")

        return solution, assignment_matrix
    else:
        print("   ✗ No optimal solution found")
        return None, None

solution, assignment_matrix = solve_concrete_example()

STATIC TRUCK APPOINTMENT SYSTEM - MATHEMATICAL OPTIMIZATION

1. Building MIP model...
   - Decision variables: 12 assignment variables
   - Gate variables: 6 gate assignment variables
   - Deviation variables: 8 deviation variables

2. Solving optimization problem...


In [ ]:
# Visualization of results
def visualize_solution(solution, assignment_matrix):
    """Create comprehensive visualizations of the solution"""
    
    if solution is None:
        print("No solution to visualize")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Static Truck Appointment System - Optimization Results', fontsize=16, fontweight='bold')
    
    # 1. Assignment Matrix Heatmap
    ax1 = axes[0, 0]
    sns.heatmap(assignment_matrix, annot=True, cmap='Blues', cbar=False,
               xticklabels=[f'Slot {t.id}' for t in time_slots],
               yticklabels=[f'Req {r.id}' for r in requests],
               ax=ax1)
    ax1.set_title('Assignment Matrix', fontweight='bold')
    ax1.set_xlabel('Time Slots')
    ax1.set_ylabel('Truck Requests')
    
    # 2. Request Deviations
    ax2 = axes[0, 1]
    request_ids = [r.id for r in requests]
    preferred_times = [r.preferred_time for r in requests]
    assigned_slots = [solution['assignments'][r.id] for r in requests]
    deviations = [abs(assigned_slots[i] - preferred_times[i]) for i in range(len(requests))]
    
    x = np.arange(len(request_ids))
    width = 0.25
    
    ax2.bar(x - width, preferred_times, width, label='Preferred', alpha=0.7, color='lightblue')
    ax2.bar(x, assigned_slots, width, label='Assigned', alpha=0.7, color='lightgreen')
    ax2.bar(x + width, deviations, width, label='Deviation', alpha=0.7, color='lightcoral')
    
    ax2.set_xlabel('Request ID')
    ax2.set_ylabel('Time Slot / Deviation')
    ax2.set_title('Request Assignments and Deviations', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(request_ids)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Gate Utilization
    ax3 = axes[1, 0]
    gate_utilization = np.zeros((len(gates), len(time_slots)))
    for g, t in product(gates, time_slots):
        gate_utilization[g.id, t.id] = solution['gate_assignments'][(g.id, t.id)]
    
    sns.heatmap(gate_utilization, annot=True, cmap='Greens', fmt='g',
               xticklabels=[f'Slot {t.id}' for t in time_slots],
               yticklabels=[f'Gate {g.id}' for g in gates],
               ax=ax3)
    ax3.set_title('Gate Utilization Matrix', fontweight='bold')
    ax3.set_xlabel('Time Slots')
    ax3.set_ylabel('Processing Gates')
    
    # 4. Time Slot Capacity Utilization
    ax4 = axes[1, 1]
    slot_capacities = [t.capacity for t in time_slots]
    slot_assignments = []
    for t in time_slots:
        assigned = sum(1 for r in requests if solution['assignments'][r.id] == t.id)
        slot_assignments.append(assigned)
    
    x = np.arange(len(time_slots))
    width = 0.35
    
    ax4.bar(x - width/2, slot_capacities, width, label='Capacity', alpha=0.7, color='skyblue')
    ax4.bar(x + width/2, slot_assignments, width, label='Assigned', alpha=0.7, color='orange')
    
    ax4.set_xlabel('Time Slots')
    ax4.set_ylabel('Number of Trucks')
    ax4.set_title('Time Slot Capacity Utilization', fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels([f'Slot {t.id}' for t in time_slots])
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Add utilization percentages on bars
    for i, (cap, assigned) in enumerate(zip(slot_capacities, slot_assignments)):
        utilization = assigned / cap * 100 if cap > 0 else 0
        ax4.text(i + width/2, assigned + 0.1, f'{utilization:.0f}%', 
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "=" * 60)
    print("SOLUTION SUMMARY STATISTICS")
    print("=" * 60)
    
    total_deviation = sum(abs(solution['assignments'][r.id] - r.preferred_time) for r in requests)
    weighted_deviation = sum(r.weight * abs(solution['assignments'][r.id] - r.preferred_time) for r in requests)
    
    print(f"\nDeviation Analysis:")
    print(f"  Total unweighted deviation: {total_deviation}")
    print(f"  Total weighted deviation: {weighted_deviation:.2f}")
    print(f"  Average deviation per request: {total_deviation/len(requests):.2f}")
    
    print(f"\nCapacity Utilization:")
    for t in time_slots:
        assigned = sum(1 for r in requests if solution['assignments'][r.id] == t.id)
        utilization = assigned / t.capacity * 100
        print(f"  Slot {t.id}: {assigned}/{t.capacity} ({utilization:.1f}% utilized)")
    
    print(f"\nGate Balance Analysis:")
    for t in time_slots:
        gate_loads = [solution['gate_assignments'][(g.id, t.id)] for g in gates]
        max_load = max(gate_loads)
        min_load = min(gate_loads)
        imbalance = max_load - min_load
        print(f"  Slot {t.id}: Gate loads {gate_loads}, imbalance = {imbalance}")

visualize_solution(solution, assignment_matrix)

No solution to visualize


In [ ]:
# Sensitivity Analysis
def perform_sensitivity_analysis():
    """Perform sensitivity analysis on key parameters"""
    
    print("\n" + "=" * 60)
    print("SENSITIVITY ANALYSIS")
    print("=" * 60)
    
    # Test different alpha (deviation weight) values
    alpha_values = [0.5, 1.0, 1.5, 2.0, 2.5]
    results = []
    
    print("\n1. Testing different deviation penalty weights (alpha):")
    print("   Alpha | Total Penalty | Deviation | Avg Deviation")
    print("   " + "-" * 45)
    
    for alpha in alpha_values:
        model = StaticTruckAppointmentMIP(requests, time_slots, gates, alpha=alpha, beta=0.5)
        model.build_model()
        
        if model.solve():
            sol = model.solution
            total_dev = sum(abs(sol['assignments'][r.id] - r.preferred_time) for r in requests)
            avg_dev = total_dev / len(requests)
            
            results.append({
                'alpha': alpha,
                'total_penalty': sol['total_penalty'],
                'total_deviation': total_dev,
                'avg_deviation': avg_dev
            })
            
            print(f"   {alpha:>5.1f} | {sol['total_penalty']:>11.2f} | {total_dev:>9} | {avg_dev:>11.2f}")
        else:
            print(f"   {alpha:>5.1f} | {'No solution':>11} | {'N/A':>9} | {'N/A':>11}")
    
    # Test different beta (gate imbalance) values
    beta_values = [0.0, 0.25, 0.5, 0.75, 1.0]
    
    print("\n2. Testing different gate imbalance weights (beta):")
    print("   Beta | Total Penalty | Deviation | Gate Balance")
    print("   " + "-" * 45)
    
    for beta in beta_values:
        model = StaticTruckAppointmentMIP(requests, time_slots, gates, alpha=1.0, beta=beta)
        model.build_model()
        
        if model.solve():
            sol = model.solution
            total_dev = sum(abs(sol['assignments'][r.id] - r.preferred_time) for r in requests)
            
            # Calculate gate balance metric
            gate_imbalances = []
            for t in time_slots:
                gate_loads = [sol['gate_assignments'][(g.id, t.id)] for g in gates]
                imbalance = max(gate_loads) - min(gate_loads)
                gate_imbalances.append(imbalance)
            avg_imbalance = sum(gate_imbalances) / len(gate_imbalances)
            
            print(f"   {beta:>4.2f} | {sol['total_penalty']:>11.2f} | {total_dev:>9} | {avg_imbalance:>11.2f}")
        else:
            print(f"   {beta:>4.2f} | {'No solution':>11} | {'N/A':>9} | {'N/A':>11}")
    
    # Visualize sensitivity results
    if results:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Alpha sensitivity plot
        alphas = [r['alpha'] for r in results]
        penalties = [r['total_penalty'] for r in results]
        deviations = [r['total_deviation'] for r in results]
        
        ax1_twin = ax1.twinx()
        ax1.plot(alphas, penalties, 'b-o', label='Total Penalty')
        ax1_twin.plot(alphas, deviations, 'r-s', label='Total Deviation')
        ax1.set_xlabel('Alpha (Deviation Weight)')
        ax1.set_ylabel('Total Penalty', color='b')
        ax1_twin.set_ylabel('Total Deviation', color='r')
        ax1.set_title('Sensitivity to Deviation Weight', fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Request priority impact
        ax2.bar(range(len(requests)), [r.weight for r in requests], alpha=0.7, color='lightgreen')
        ax2.set_xlabel('Request ID')
        ax2.set_ylabel('Priority Weight')
        ax2.set_title('Request Priority Weights', fontweight='bold')
        ax2.set_xticks(range(len(requests)))
        ax2.set_xticklabels([f'Req {r.id}' for r in requests])
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

perform_sensitivity_analysis()


SENSITIVITY ANALYSIS

1. Testing different deviation penalty weights (alpha):
   Alpha | Total Penalty | Deviation | Avg Deviation
   ---------------------------------------------
     0.5 | No solution |       N/A |         N/A
     1.0 | No solution |       N/A |         N/A
     1.5 | No solution |       N/A |         N/A
     2.0 | No solution |       N/A |         N/A
     2.5 | No solution |       N/A |         N/A

2. Testing different gate imbalance weights (beta):
   Beta | Total Penalty | Deviation | Gate Balance
   ---------------------------------------------
   0.00 | No solution |       N/A |         N/A
   0.25 | No solution |       N/A |         N/A
   0.50 | No solution |       N/A |         N/A
   0.75 | No solution |       N/A |         N/A
   1.00 | No solution |       N/A |         N/A


### Why this Tier exists vs earlier Tiers
This Tier 1 provides the mathematical foundation for the Static Truck Appointment System problem:

- **Optimal solutions**: Guarantees mathematically optimal assignments
- **Formal problem definition**: Precise mathematical formulation with constraints
- **Benchmark for comparison**: Provides baseline for evaluating heuristic methods
- **Sensitivity analysis**: Understanding of parameter impacts on solutions
- **Theoretical foundation**: Basis for understanding problem complexity and structure

### Pros / Cons vs earlier Tiers
**Pros:**
- Guarantees optimal solutions (when solvable)
- Provides mathematical rigor and formal problem definition
- Enables sensitivity analysis and parameter tuning
- Serves as benchmark for heuristic evaluation
- Clear understanding of problem structure and constraints

**Cons:**
- Computationally expensive for large problem instances
- May not scale well with many requests and time slots
- Requires specialized optimization software (pulp solver)
- Less flexible for dynamic or real-time applications
- May struggle with complex constraints or non-linear objectives

### When to use this Tier
- **Small to medium problem instances** where optimality is important
- **Planning and strategic decisions** where computational time is acceptable
- **Benchmark development** for evaluating heuristic methods
- **Parameter sensitivity analysis** to understand system behavior
- **Academic and research settings** requiring mathematical rigor
- **Systems with well-defined constraints** and linear objectives